# Chapter 2: Overview of Text Classification
**Module 03 – Deep Learning for Text with PyTorch**

> *Instructor: Shubham Jain, Data Scientist*

## 2.1 Text Classification Defined

Text classification assigns **labels** to pieces of text, giving structure to unstructured data.

| Type | Description | Example |
|---|---|---|
| **Binary** | Two categories | Spam vs Not Spam |
| **Multi-class** | Multiple exclusive categories | News: Politics / Sports / Tech |
| **Multi-label** | Multiple tags per text | Book genres: Action + Fantasy |

## 2.2 From Tokens to Numbers: Encoding

Neural networks need **numbers**, not words. We convert tokens to indices:

```
"King" → 1
"Queen" → 2
```

Then we use **word embeddings** to convert indices into dense vector representations.

## 2.3 Word Embeddings in PyTorch

`nn.Embedding` maps integer indices to dense vectors:
- Each unique word → a learned vector of fixed size
- Similar words end up with **similar vectors** (e.g., "King" and "Queen" are close)
- Captures **semantic relationships**

In [ ]:
import torch
import torch.nn as nn

# Vocabulary: word → index
vocab = {'<PAD>': 0, 'The': 1, 'cat': 2, 'sat': 3, 'on': 4, 'the': 5, 'mat': 6}

# Embedding layer: vocab_size=7, embedding_dim=4
embedding = nn.Embedding(num_embeddings=len(vocab), embedding_dim=4)

# Convert sentence to index tensor
sentence = ['The', 'cat', 'sat', 'on', 'the', 'mat']
indices = torch.tensor([vocab[w] for w in sentence])

# Get embeddings
embedded = embedding(indices)
print("Sentence:", sentence)
print("Indices: ", indices.tolist())
print("Embeddings shape:", embedded.shape)  # (6, 4)
print("Embedding for 'cat':")
print(embedded[1].detach())

## 2.4 Text Classification Model with Embeddings

In [ ]:
class TextClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_classes):
        super(TextClassifier, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.fc1 = nn.Linear(embed_dim, 64)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.3)
        self.fc2 = nn.Linear(64, num_classes)

    def forward(self, x):
        # x: (batch_size, seq_length)
        embedded = self.embedding(x)         # (batch, seq, embed_dim)
        pooled = embedded.mean(dim=1)        # Average pooling: (batch, embed_dim)
        out = self.fc1(pooled)
        out = self.relu(out)
        out = self.dropout(out)
        out = self.fc2(out)
        return out

# Create model
VOCAB_SIZE = 10000
EMBED_DIM  = 64
NUM_CLASSES = 3  # e.g., Politics, Sports, Tech

model = TextClassifier(VOCAB_SIZE, EMBED_DIM, NUM_CLASSES)

# Test with a batch of 8 sequences of length 20
batch = torch.randint(0, VOCAB_SIZE, (8, 20))
print("Output shape:", model(batch).shape)  # (8, 3)

## 2.5 Padding Sequences

Texts have different lengths. We **pad** shorter sequences with zeros to create uniform batches.

In [ ]:
from torch.nn.utils.rnn import pad_sequence

# Variable-length sequences
seq1 = torch.tensor([1, 2, 3, 4, 5])
seq2 = torch.tensor([1, 2])
seq3 = torch.tensor([1, 2, 3])

# Pad to the length of the longest sequence
padded = pad_sequence([seq1, seq2, seq3], batch_first=True, padding_value=0)
print("Padded batch:\n", padded)
print("Shape:", padded.shape)  # (3, 5)

## Summary

| Concept | Key Detail |
|---|---|
| Text classification | Binary, multi-class, multi-label |
| `nn.Embedding` | Turns word indices into dense vectors |
| Mean pooling | Averages token embeddings to get sentence vector |
| Padding | Makes variable-length sequences uniform |